<a href="https://colab.research.google.com/github/neerajmahajan83/pythonTutorial/blob/main/day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What is a Metaclass?

In Python, a metaclass is the class of a class. Just as an object is an instance of a class, a class is an instance of a metaclass. Metaclasses allow you to define how classes are created, essentially providing a hook into the class creation process. This can be useful for automatic registration of classes, adding methods to classes, or validating class definitions, among other advanced use cases.

In [ ]:
# Define a simple metaclass
class MyMeta(type):
    def __new__(mcs, name, bases, namespace):
        # 'mcs' is the metaclass itself (MyMeta)
        # 'name' is the name of the class being created (e.g., 'MyClass')
        # 'bases' is a tuple of base classes
        # 'namespace' is the dictionary of class attributes and methods

        print(f"\nCreating class: {name}")
        print(f"  Bases: {bases}")
        print(f"  Namespace: {namespace}")

        # Add a custom attribute to the class being created
        namespace['custom_attribute'] = 'Hello from Metaclass!'

        # Add a custom method to the class being created
        def custom_method(self):
            return f"This is a custom method for {self.name} with attribute: {self.custom_attribute}"
        namespace['custom_method'] = custom_method

        # Call the parent's __new__ to actually create the class object
        return super().__new__(mcs, name, bases, namespace)

    def __init__(cls, name, bases, namespace):
        # 'cls' is the newly created class object
        print(f"Initializing class: {name}")
        # Corrected: Call super().__init__ with only 'cls'
        super().__init__(cls)


### Using the Metaclass

To use a metaclass, you specify it in the class definition using the `metaclass` keyword argument.

In [ ]:
# Define a class using MyMeta as its metaclass
class MyClass(metaclass=MyMeta):
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hi, I am {self.name}"

# Instantiate the class
obj = MyClass("Alice")

# Access attributes and methods added by the metaclass
print(f"\nObject name: {obj.name}")
print(f"Object custom attribute: {obj.custom_attribute}") # Added by metaclass
print(f"Object greet method: {obj.greet()}")
print(f"Object custom method: {obj.custom_method()}") # Added by metaclass

# Verify the type of MyClass and obj
print(f"\nType of MyClass: {type(MyClass)}")
print(f"Type of obj: {type(obj)}")


Creating class: MyClass
  Bases: ()
  Namespace: {'__module__': '__main__', '__qualname__': 'MyClass', '__init__': <function MyClass.__init__ at 0x79f62f0aa160>, 'greet': <function MyClass.greet at 0x79f62f0a98a0>}
Initializing class: MyClass

Object name: Alice
Object custom attribute: Hello from Metaclass!
Object greet method: Hi, I am Alice
Object custom method: This is a custom method for Alice with attribute: Hello from Metaclass!

Type of MyClass: <class '__main__.MyMeta'>
Type of obj: <class '__main__.MyClass'>


### Example 2: Automatic Class Registration with Metaclasses

This example shows how a metaclass can be used to automatically register all classes that use it into a central dictionary. This is a common pattern for plugin systems or command dispatchers.

In [ ]:
# A dictionary to hold all registered classes
REGISTERED_COMMANDS = {}

class CommandMeta(type):
    """Metaclass that automatically registers classes."""
    def __new__(mcs, name, bases, namespace):
        # Create the class object using the default type.__new__
        cls = super().__new__(mcs, name, bases, namespace)

        # If the class has a 'command_name' attribute and it's not the base Command class itself,
        # register it.
        if hasattr(cls, 'command_name') and cls.command_name and name != 'Command':
            REGISTERED_COMMANDS[cls.command_name] = cls
            print(f"Registered command: {cls.command_name} -> {cls.__name__}")
        return cls

class Command(metaclass=CommandMeta):
    """Base Command class."""
    command_name = None # This will be set by subclasses

    def execute(self, *args, **kwargs):
        raise NotImplementedError("Subclasses must implement 'execute' method")

class GreetCommand(Command):
    """A command to greet a user."""
    command_name = 'greet'

    def execute(self, name):
        return f"Hello, {name}!"

class FarewellCommand(Command):
    """A command to bid farewell."""
    command_name = 'farewell'

    def execute(self, name):
        return f"Goodbye, {name}! Have a great day!"

# Define another command that won't be registered because command_name is None
class UnregisteredCommand(Command):
    # command_name is None, so it won't be registered
    def execute(self):
        return "This command is not registered."

print("\n--- Verifying registered commands ---")
print(f"All registered commands: {REGISTERED_COMMANDS}")

print("\n--- Executing registered commands ---")
if 'greet' in REGISTERED_COMMANDS:
    greet_instance = REGISTERED_COMMANDS['greet']()
    print(greet_instance.execute("Alice"))

if 'farewell' in REGISTERED_COMMANDS:
    farewell_instance = REGISTERED_COMMANDS['farewell']()
    print(farewell_instance.execute("Bob"))

# Try to execute the unregistered command (it won't be in the registry)
if 'unregistered' not in REGISTERED_COMMANDS:
    print("\n'unregistered' command is not in the registry, as expected.")


Registered command: greet -> GreetCommand
Registered command: farewell -> FarewellCommand

--- Verifying registered commands ---
All registered commands: {'greet': <class '__main__.GreetCommand'>, 'farewell': <class '__main__.FarewellCommand'>}

--- Executing registered commands ---
Hello, Alice!
Goodbye, Bob! Have a great day!

'unregistered' command is not in the registry, as expected.


### Simplest Metaclass Example

This example demonstrates the absolute core concept: a metaclass is called when a class is defined, giving it a chance to do something during that definition process.

In [ ]:
class SimpleMeta(type):
    def __new__(mcs, name, bases, namespace):
        print(f"\n>>> SimpleMeta is creating class: {name}")
        # Call the normal class creation process
        return super().__new__(mcs, name, bases, namespace)

class MySimpleClass(metaclass=SimpleMeta):
    value = 10

    def __init__(self):
        print(f"  MySimpleClass object initialized.")

    def get_value(self):
        return self.value

print("\n--- Creating an instance of MySimpleClass ---")
obj = MySimpleClass()
print(f"Object's value: {obj.get_value()}")
print(f"Type of MySimpleClass: {type(MySimpleClass)}")


>>> SimpleMeta is creating class: MySimpleClass

--- Creating an instance of MySimpleClass ---
  MySimpleClass object initialized.
Object's value: 10
Type of MySimpleClass: <class '__main__.SimpleMeta'>


In [ ]:
class Outer:
  def __init__(self):
    self.name = "Outer Class"

  class Inner:
    def __init__(self):
      self.name = "Inner Class"

    def display(self):
      print("This is the inner class")

outer = Outer()
print(outer.name)

Outer Class


### Explanation of the Outer and Inner Classes

In Python, you can define classes inside other classes. These are known as **nested classes** or **inner classes**. In your example:

*   **`Outer` Class**: This is the enclosing class. It has an `__init__` method that sets its own `name` attribute to "Outer Class".
*   **`Inner` Class**: This class is defined *inside* the `Outer` class. It also has an `__init__` method that sets its `name` attribute to "Inner Class", and a `display` method that prints a message.

### Advantages of Nested Classes

Nested classes offer several advantages in terms of organization, encapsulation, and logical grouping:

1.  **Encapsulation and Information Hiding**: The inner class can be tightly coupled with the outer class, often used to implement functionality that is logically part of the outer class but shouldn't be exposed directly at the top level.
2.  **Logical Grouping**: When a class is exclusively used by another class, nesting it can improve code readability and maintainability by grouping related components together.
3.  **Namespace Management**: Nested classes help avoid polluting the global namespace. The inner class is scoped within the outer class.
4.  **Implementing Design Patterns**: They are useful in design patterns like the Iterator pattern (where an iterator class might be nested inside a collection class) or when creating Helper Classes that are specific to the outer class's implementation.

In your example, `Inner` is a member of `Outer`. You can access the `Inner` class through an instance of `Outer` (though not explicitly shown in your current instantiation) or directly via `Outer.Inner` if it's meant to be a static inner class. The line `outer = Outer()` creates an instance of the outer class, and `print(outer.name)` correctly prints the name of the outer class instance.

### Python Encapsulation Example

Encapsulation is one of the fundamental principles of object-oriented programming (OOP). It describes the bundling of data with the methods that operate on that data, or the restricting of direct access to some of an object's components.

In [ ]:
class BankAccount:
    def __init__(self, account_holder, initial_balance=0):
        # Private attributes (conventionally indicated by a single leading underscore)
        self.__account_holder = account_holder
        self.__balance = initial_balance

    # Public method to deposit money
    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            print(f"Deposited ${amount}. New balance: ${self.__balance}")
        else:
            print("Deposit amount must be positive.")

    # Public method to withdraw money
    def withdraw(self, amount):
        if 0 < amount <= self.__balance:
            self.__balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self.__balance}")
        elif amount > self.__balance:
            print("Insufficient funds.")
        else:
            print("Withdrawal amount must be positive.")

    # Public method to get the account holder's name
    def get_account_holder(self):
        return self.__account_holder

    # Public method to get the current balance
    def get_balance(self):
        return self.__balance

# Create an instance of BankAccount
my_account = BankAccount("Alice Wonderland", 1000)

# Interact with the account using public methods
print(f"Account Holder: {my_account.get_account_holder()}")
print(f"Current Balance: ${my_account.get_balance()}")

my_account.deposit(500)
my_account.withdraw(200)
my_account.withdraw(1500) # This should fail due to insufficient funds
my_account.deposit(-100) # This should fail due to negative amount

print(f"Final Balance: ${my_account.get_balance()}")

# Attempting to directly access or modify 'private' attributes (discouraged but technically possible in Python)
# print(my_account.__balance) # This would raise an AttributeError
# print(my_account._BankAccount__balance) # This is 'name mangling' and can be used to access, but breaks encapsulation


Account Holder: Alice Wonderland
Current Balance: $1000
Deposited $500. New balance: $1500
Withdrew $200. New balance: $1300
Insufficient funds.
Deposit amount must be positive.
Final Balance: $1300


### Explanation of Encapsulation

In the `BankAccount` example:

*   **Private Attributes**: `__account_holder` and `__balance` are considered private attributes. In Python, this is achieved by prefixing the attribute name with two underscores (`__`). This triggers Python's name mangling, making it harder (though not impossible) to access them directly from outside the class. This convention signals to other developers that these attributes should not be directly modified.

*   **Public Methods**: `deposit`, `withdraw`, `get_account_holder`, and `get_balance` are public methods. These methods provide the only sanctioned way to interact with the private data of the `BankAccount` object.

**Key benefits demonstrated:**

1.  **Data Hiding**: The internal state (`__balance`) is hidden from direct external access. You can't just set `my_account.__balance = -100`, for instance.
2.  **Controlled Access**: Data can only be accessed or modified through the defined public methods. This allows the class to enforce rules (e.g., deposit amount must be positive, withdrawal amount cannot exceed balance) and maintain the integrity of its data.
3.  **Flexibility and Maintainability**: If the internal representation of `__balance` needs to change (e.g., from an integer to a `Decimal` object for precision), you only need to modify the methods within the `BankAccount` class. External code using `deposit()` or `get_balance()` would not need to change, as the interface remains consistent.

### Python Polymorphism Example

Polymorphism, meaning "many forms," is another fundamental concept in object-oriented programming. It allows objects of different classes to be treated as objects of a common type. In Python, polymorphism is often achieved through method overriding and duck typing.

In [ ]:
class Animal:
    def speak(self):
        raise NotImplementedError("Subclass must implement abstract method")

class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

class Duck(Animal):
    def speak(self):
        return "Quack!"

# A function that can take any object that has a 'speak' method
def make_animal_speak(animal):
    print(animal.speak())

# Create instances of different animal classes
dog = Dog()
cat = Cat()
duck = Duck()

print("--- Demonstrating Polymorphism ---")
# Call the function with different animal objects
make_animal_speak(dog)
make_animal_speak(cat)
make_animal_speak(duck)

# Another way to show polymorphism: iterating through a list of different objects
animals = [Dog(), Cat(), Duck()]

print("\n--- Iterating through a list of diverse animal objects ---")
for animal in animals:
    make_animal_speak(animal)


--- Demonstrating Polymorphism ---
Woof!
Meow!
Quack!

--- Iterating through a list of diverse animal objects ---
Woof!
Meow!
Quack!


### Explanation of Polymorphism

In the `Polymorphism` example:

*   **Base Class (`Animal`)**: We define a base class `Animal` with a `speak` method. This method raises a `NotImplementedError`, indicating that subclasses *must* provide their own implementation of `speak`. This establishes a common interface.

*   **Derived Classes (`Dog`, `Cat`, `Duck`)**: Each of these classes inherits from `Animal` and provides its own unique implementation of the `speak` method. This is an example of **method overriding**.

*   **Polymorphic Function (`make_animal_speak`)**: The `make_animal_speak` function takes an `animal` object as an argument and calls its `speak()` method. The beauty of polymorphism is that this function doesn't need to know *what specific type* of animal it's dealing with (Dog, Cat, or Duck). As long as the object has a `speak()` method, the function will work correctly. This is also an illustration of **duck typing** (if it walks like a duck and quacks like a duck, it's a duck) – Python cares more about *what an object can do* than *what type it is*.

*   **List of Diverse Objects**: We create a list `animals` containing instances of `Dog`, `Cat`, and `Duck`. When we iterate through this list and call `make_animal_speak` for each item, the correct `speak` method (specific to each animal's class) is invoked automatically.

**Key benefits demonstrated:**

1.  **Code Reusability**: The `make_animal_speak` function can be used with any object that conforms to the `speak` interface, reducing the need for type-specific conditional logic.
2.  **Flexibility and Extensibility**: You can add new `Animal` subclasses (e.g., `Cow`, `Pig`) with their own `speak` methods without modifying the `make_animal_speak` function or the existing `animals` list iteration. The system will automatically accommodate the new types.
3.  **Maintainability**: Changes to the implementation of a specific animal's `speak` method only affect that class, not the higher-level functions that use the polymorphic interface.

### Python Inheritance Example

Inheritance is a fundamental concept in object-oriented programming that allows a new class (subclass or derived class) to inherit properties and behaviors (methods and attributes) from an existing class (superclass or base class). This promotes code reusability and establishes a natural "is-a" relationship between classes.

In [ ]:
class Vehicle:
    """Base class for all vehicles."""
    def __init__(self, brand, model, year):
        self.brand = brand
        self.model = model
        self.year = year

    def display_info(self):
        return f"Brand: {self.brand}, Model: {self.model}, Year: {self.year}"

    def start_engine(self):
        return "Engine started."

class Car(Vehicle):
    """Derived class for cars, inheriting from Vehicle."""
    def __init__(self, brand, model, year, num_doors):
        # Call the constructor of the base class (Vehicle)
        super().__init__(brand, model, year)
        self.num_doors = num_doors

    def drive(self):
        return "Car is driving."

    # Override the display_info method from the base class
    def display_info(self):
        base_info = super().display_info()
        return f"{base_info}, Doors: {self.num_doors}"

class Motorcycle(Vehicle):
    """Derived class for motorcycles, inheriting from Vehicle."""
    def __init__(self, brand, model, year, has_sidecar):
        super().__init__(brand, model, year)
        self.has_sidecar = has_sidecar

    def ride(self):
        return "Motorcycle is riding."

# Create instances of the derived classes
my_car = Car("Toyota", "Camry", 2020, 4)
my_motorcycle = Motorcycle("Harley-Davidson", "Fat Boy", 2023, False)

print("--- Car Information ---")
print(my_car.display_info()) # Calls overridden method
print(my_car.start_engine()) # Calls inherited method
print(my_car.drive())      # Calls Car's specific method

print("\n--- Motorcycle Information ---")
print(my_motorcycle.display_info()) # Calls inherited method
print(my_motorcycle.start_engine()) # Calls inherited method
print(my_motorcycle.ride())   # Calls Motorcycle's specific method


--- Car Information ---
Brand: Toyota, Model: Camry, Year: 2020, Doors: 4
Engine started.
Car is driving.

--- Motorcycle Information ---
Brand: Harley-Davidson, Model: Fat Boy, Year: 2023
Engine started.
Motorcycle is riding.


### Explanation of Inheritance

In this inheritance example:

*   **Base Class (`Vehicle`)**: This is the parent class. It defines common attributes (`brand`, `model`, `year`) and methods (`display_info`, `start_engine`) that all vehicles share.

*   **Derived Classes (`Car`, `Motorcycle`)**: These are the child classes that inherit from `Vehicle`. This means they automatically get all the public attributes and methods of the `Vehicle` class.

    *   **`Car` Class**:
        *   It uses `super().__init__(brand, model, year)` to call the `Vehicle` class's constructor and initialize the common attributes.
        *   It adds its own unique attribute (`num_doors`) and method (`drive`).
        *   It **overrides** the `display_info` method to provide a `Car`-specific representation, while still leveraging the base class's `display_info` using `super().display_info()`.

    *   **`Motorcycle` Class**:
        *   It also calls `super().__init__` to initialize base class attributes.
        *   It adds its own unique attribute (`has_sidecar`) and method (`ride`).

**Key benefits demonstrated:**

1.  **Code Reusability**: Common code for `brand`, `model`, `year`, and `start_engine` is written once in `Vehicle` and reused by `Car` and `Motorcycle`, avoiding duplication.
2.  **"Is-A" Relationship**: A `Car` "is a" `Vehicle`, and a `Motorcycle` "is a" `Vehicle`. This reflects a natural hierarchy and makes the code more organized and easier to understand.
3.  **Extensibility**: You can easily add new types of vehicles (e.g., `Truck`, `Bicycle`) by creating new classes that inherit from `Vehicle`, without modifying the existing `Vehicle` class.
4.  **Method Overriding**: Derived classes can provide their own specific implementations of methods defined in the base class, tailoring behavior while maintaining a common interface.

### Python Class Method Example

In Python, a class method is a method that is bound to the class and not the instance of the class. It can modify a class state that would apply across all instances of the class. It is defined using the `@classmethod` decorator.

In [ ]:
class MyClass:
    class_variable = "I am a class variable"

    def __init__(self, instance_variable):
        self.instance_variable = instance_variable

    def instance_method(self):
        """An instance method that operates on instance data."""
        return f"Instance method: instance_variable='{self.instance_variable}', class_variable='{self.class_variable}'"

    @classmethod
    def class_method(cls):
        """A class method that operates on class data. It receives the class itself (cls) as the first argument."""
        return f"Class method: Accessing class_variable='{cls.class_variable}'"

    @classmethod
    def create_instance_with_prefix(cls, prefix, name):
        """A class method that can create an instance of the class."""
        new_instance_name = f"{prefix}_{name}"
        return cls(new_instance_name)

# Create an instance of the class
obj1 = MyClass("value1")

# Call the instance method
print(obj1.instance_method())

# Call the class method using the class itself
print(MyClass.class_method())

# Call the class method using an instance (still refers to the class)
print(obj1.class_method())

# Demonstrate modifying a class variable using a class method (though not directly done in `class_method` itself)
print(f"\nInitial class_variable: {MyClass.class_variable}")
MyClass.class_variable = "Updated by direct class access"
print(MyClass.class_method())

# Using a class method to create a new instance
obj2 = MyClass.create_instance_with_prefix("new", "value")
print(f"New instance created by class method: {obj2.instance_variable}")
print(obj2.instance_method())


Instance method: instance_variable='value1', class_variable='I am a class variable'
Class method: Accessing class_variable='I am a class variable'
Class method: Accessing class_variable='I am a class variable'

Initial class_variable: I am a class variable
Class method: Accessing class_variable='Updated by direct class access'
New instance created by class method: new_value
Instance method: instance_variable='new_value', class_variable='Updated by direct class access'


### Explanation of Class Methods

In this `MyClass` example:

*   **`class_variable`**: This is a class-level attribute, shared by all instances of `MyClass`.

*   **`__init__(self, instance_variable)`**: This is the constructor, which initializes `instance_variable`, an attribute unique to each instance.

*   **`instance_method(self)`**: This is a regular instance method. It takes `self` (the instance) as its first argument and can access both instance-specific data (`self.instance_variable`) and class-level data (`self.class_variable`).

*   **`@classmethod` `class_method(cls)`**: This is a class method, indicated by the `@classmethod` decorator. Instead of `self`, it takes `cls` (the class itself) as its first argument. It can access and modify class-level attributes (`cls.class_variable`) but does not have direct access to instance-specific attributes without an instance object.
    *   You can call a class method using either the class name (`MyClass.class_method()`) or an instance (`obj1.class_method()`). In both cases, `cls` inside the method will refer to `MyClass`.

*   **`@classmethod` `create_instance_with_prefix(cls, prefix, name)`**: This is another common use case for class methods: factory methods. It takes the class `cls` and other arguments, then uses `cls()` to create and return a new instance of the class. This is useful for providing alternative ways to construct objects.

**Key differences and benefits:**

1.  **Binding**: Instance methods are bound to the object, while class methods are bound to the class.
2.  **First Argument**: Instance methods receive the instance (`self`), while class methods receive the class (`cls`).
3.  **Use Cases**:
    *   **Class Methods**: Ideal for operations that involve the class state (like modifying class variables), or for creating factory methods that return instances of the class with specific configurations (as shown with `create_instance_with_prefix`).
    *   **Instance Methods**: Ideal for operations that involve the state of a specific instance.

Class methods provide a way to define behavior that pertains to the class as a whole, rather than to any particular instance of the class.

### Python Class Properties Example (using `@property` decorator)

In Python, the `@property` decorator provides a way to define methods that can be accessed like attributes, giving you more control over how attributes are accessed and modified. This is a common way to implement the concept of 'properties' in Python classes, allowing you to add logic (like validation) when an attribute is read or written, without changing the way the attribute is accessed from outside the class.

In [ ]:
class Person:
    def __init__(self, name, age):
        self._name = name  # Internal attribute, conventionally prefixed with a single underscore
        self._age = None   # Initialize with None, will be set by the property setter
        self.age = age     # Use the setter for initial validation

    @property
    def name(self):
        """The 'name' property (getter method)."""
        print("Getting name...")
        return self._name

    @name.setter
    def name(self, value):
        """The 'name' property (setter method)."""
        print("Setting name...")
        if not isinstance(value, str) or not value.strip():
            raise ValueError("Name must be a non-empty string.")
        self._name = value.strip()

    @property
    def age(self):
        """The 'age' property (getter method)."""
        print("Getting age...")
        return self._age

    @age.setter
    def age(self, value):
        """The 'age' property (setter method)."""
        print("Setting age...")
        if not isinstance(value, int) or value <= 0:
            raise ValueError("Age must be a positive integer.")
        self._age = value

    @name.deleter
    def name(self):
        """The 'name' property (deleter method)."""
        print("Deleting name...")
        del self._name

# Create an instance
print("--- Creating a Person instance ---")
try:
    p = Person("Alice", 30)
    print(f"Initial name: {p.name}, age: {p.age}")
except ValueError as e:
    print(f"Error creating person: {e}")

print("\n--- Accessing properties ---")
print(f"Person's name: {p.name}") # Calls the name getter
print(f"Person's age: {p.age}")   # Calls the age getter

print("\n--- Modifying properties ---")
p.name = "Bob Smith" # Calls the name setter
p.age = 25           # Calls the age setter
print(f"Updated name: {p.name}, age: {p.age}")

print("\n--- Demonstrating validation (invalid age) ---")
try:
    p.age = -5 # This will call the setter and raise an error
except ValueError as e:
    print(f"Caught expected error: {e}")
print(f"Age remains: {p.age}")

print("\n--- Demonstrating validation (invalid name) ---")
try:
    p.name = "" # This will call the setter and raise an error
except ValueError as e:
    print(f"Caught expected error: {e}")
print(f"Name remains: {p.name}")

print("\n--- Deleting a property ---")
try:
    del p.name # Calls the name deleter
    print(f"Attempting to access name after deletion: {p.name}")
except AttributeError as e:
    print(f"Caught expected error after deletion: {e}")


### Explanation of Class Properties

In this `Person` example:

*   **Internal Attributes**: `_name` and `_age` are the actual attributes storing the data. The single leading underscore (`_`) is a convention in Python to indicate that these are "protected" or internal attributes that should not be accessed directly from outside the class. This hints that users should interact with them through the defined properties.

*   **`@property` Decorator**: This decorator is placed above a method (`name`, `age`) to turn it into a "getter" for a property. When you access `p.name` or `p.age`, this method is automatically called.

*   **`@<property_name>.setter` Decorator**: This decorator defines the "setter" method for a property. For example, `@name.setter` is used for the method that gets called when you assign a value to `p.name` (e.g., `p.name = "Bob"`). Inside the setter, you can add logic like type checking or value validation.

*   **`@<property_name>.deleter` Decorator**: This decorator defines the "deleter" method for a property. For example, `@name.deleter` is used for the method that gets called when you use `del p.name`.

**Key benefits demonstrated:**

1.  **Controlled Access**: Properties allow you to control how attributes are read, written, and deleted. You can add custom logic at each step.
2.  **Validation**: As shown with `age` and `name`, you can enforce rules (e.g., `age` must be positive, `name` must be a non-empty string) when an attribute is set, ensuring data integrity.
3.  **Readability/Maintainability**: From the outside, `p.name` and `p.age` look and behave like regular attributes. This means you can add complex logic behind the scenes (getters, setters) without changing the external interface of your class, making code easier to refactor and maintain.
4.  **"Getter/Setter" without explicit calls**: Unlike languages like Java that require `getName()` and `setName()` methods, Python's properties allow you to achieve the same control with a more Pythonic, attribute-like syntax.